# EG-VAN three-arm diagnostic (exploratory only)
This workflow is `EXPLORATORY_ONLY_NONCLINICAL_NOT_PAPER_REPRODUCTION`; the cohort is `NOT_APPROVED`. It is not clinical validation, an exact paper reproduction, or external validation. The diagnostic runner may inspect only the frozen train and validation partitions. Do not copy calibration/test images or checkpoints into this workflow. No package installation, dataset preparation, general exploratory preflight, or training occurs by default.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys
PROJECT = Path('/content/drive/.shortcut-targets-by-id/1O_G4-Fo1RGgNekg9RSyjdipPkUnylKh4/EG-VAN').resolve()
assert (PROJECT / 'src/nine_class_three_arm_diagnostic.py').is_file(), f'Current EG-VAN project not found: {PROJECT}'
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / 'src'))
print('Project root:', PROJECT)

In [ ]:
import platform
import numpy as np
import pandas as pd
import torch
import torchvision
import cv2
versions = {
    'python': platform.python_version(),
    'pytorch': torch.__version__,
    'torchvision': torchvision.__version__,
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'opencv': cv2.__version__,
    'cuda_runtime': torch.version.cuda,
    'cuda_available': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(versions)
print('No packages are installed or upgraded by this notebook.')

## Read-only train/validation preflight
This explicitly calls the isolated diagnostic preflight. It verifies source metadata and decodes only one representative image per class in train and validation. It does not use the general four-partition preflight.

In [ ]:
from nine_class_three_arm_diagnostic import ALLOWED_IMAGE_PARTITIONS, RESEARCH_STATUS, preflight_diagnostic
assert ALLOWED_IMAGE_PARTITIONS == frozenset({'train', 'validation'})
preflight = preflight_diagnostic(PROJECT)
assert preflight['research_status'] == RESEARCH_STATUS
assert set(preflight['checked_image_partitions']) == ALLOWED_IMAGE_PARTITIONS
print(preflight)

## Training is disabled by default
The only training entry point here is `src/nine_class_three_arm_diagnostic.py`. It runs the fixed A/B/C five-epoch diagnostic with shared initialization, train-only optimization, validation-only evaluation, diagnostic LR `0.0001`, mixed precision off, and the original frozen base config unchanged. Set the flag only after separately authorizing training and reviewing the preflight.

In [ ]:
import subprocess
RUN_DIAGNOSTIC = False
RUNS_ROOT = PROJECT / 'analysis/nine_class_three_arm_diagnostics'
if RUN_DIAGNOSTIC:
    if not torch.cuda.is_available():
        raise RuntimeError('The diagnostic requires CUDA; CPU fallback is not permitted.')
    command = [
        sys.executable, 'src/nine_class_three_arm_diagnostic.py',
        '--root', str(PROJECT), '--runs-root', str(RUNS_ROOT),
    ]
    print('Starting explicitly authorized exploratory diagnostic:', command)
    subprocess.run(command, check=True)
else:
    print('Training remains disabled (RUN_DIAGNOSTIC=False).')

## Explicit resume only
Resume reuses the same experiment identity, shared initialization, split, configuration, and sample-order hashes. It is not automatic. Fill in an existing interrupted experiment ID and set `RUN_RESUME=True` only after a separate authorization. Failed arms cannot be resumed by the runner.

In [ ]:
RUN_RESUME = False
RESUME_EXPERIMENT_ID = ''
if RUN_RESUME:
    if not RESUME_EXPERIMENT_ID:
        raise ValueError('Set the exact interrupted experiment ID before resuming.')
    if not torch.cuda.is_available():
        raise RuntimeError('The diagnostic requires CUDA; CPU fallback is not permitted.')
    command = [
        sys.executable, 'src/nine_class_three_arm_diagnostic.py',
        '--root', str(PROJECT), '--runs-root', str(RUNS_ROOT),
        '--resume-experiment', RESUME_EXPERIMENT_ID,
    ]
    print('Resuming explicitly authorized exploratory diagnostic:', command)
    subprocess.run(command, check=True)
else:
    print('Resume remains disabled (RUN_RESUME=False).')